# Hướng dẫn chạy dự án trên Google Colab

Chọn **một** trong hai cách bên dưới:

## Cách A — Clone từ GitHub/GitLab (khuyến nghị trên Colab)
```bash
git clone https://github.com/vuthainguyen1602/IEEECS_CPS_2026_paper.git
```
Thư mục sau clone sẽ là `IEEECS_CPS_2026_paper/`, code chạy trong `IEEECS_CPS_2026_paper/paper_1/`.

## Cách B — Dùng Google Drive
Upload repo lên Drive, rồi mount Drive và trỏ `PROJECT_DIR` tới `.../paper_1`.

## Dữ liệu CICIDS2017
Đặt 8 file CSV vào `paper_1/data/raw/` hoặc một thư mục riêng, rồi truyền `--input-dir`.

> Colab **không cần Java/Spark**. Pipeline xử lý dữ liệu dùng pandas.

## Bước 1A: Clone repo (GitHub/GitLab)

Chạy cell này nếu bạn **clone trực tiếp trên Colab**.
Bỏ qua bước 1B nếu không dùng Google Drive.

In [ ]:
# GitHub
!git clone https://github.com/vuthainguyen1602/IEEECS_CPS_2026_paper.git

# GitLab: bỏ comment và sửa URL cho đúng
# !git clone https://gitlab.com/username/IEEECS_CPS_2026_paper.git

## Bước 1B: Mount Google Drive (tùy chọn)

Chỉ chạy nếu bạn upload repo lên Drive thay vì clone ở bước 1A.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Bước 2: Vào thư mục dự án và kiểm tra dữ liệu

Sửa `PROJECT_DIR` nếu clone từ GitLab hoặc lưu repo ở vị trí khác.
Sửa `RAW_DATA_DIR` nếu CSV nằm ngoài `paper_1/data/raw/`.

In [ ]:
import os
from pathlib import Path

# Sau git clone trên Colab:
PROJECT_DIR = "/content/IEEECS_CPS_2026_paper/paper_1"

# Nếu dùng Google Drive, sửa thành:
# PROJECT_DIR = "/content/drive/MyDrive/IEEECS_CPS_2026_paper/paper_1"

RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"  # hoặc: "/content/drive/MyDrive/ids-2017"

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy {PROJECT_DIR}. "
        "Hãy chạy bước clone hoặc sửa PROJECT_DIR cho đúng tên thư mục sau clone."
    )

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Raw CSV directory:", RAW_DATA_DIR)
print("CSV files found:", len(list(Path(RAW_DATA_DIR).glob("*.csv"))))
!ls -la

## Bước 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -r requirements.txt

## Bước 4: Chuẩn bị dữ liệu

Script đọc CSV từ `RAW_DATA_DIR`, gộp/làm sạch, rồi lưu Parquet vào `data/`.

In [ ]:
!python code/data_preparation.py --input-dir {RAW_DATA_DIR}

# Nếu CSV ở thư mục khác:
# !python code/data_preparation.py --input-dir "/content/drive/MyDrive/ids-2017"

## Bước 5: Huấn luyện mô hình

Sau khi có `data/train_data.parquet` và `data/test_data.parquet`, chạy huấn luyện teacher và student.

In [ ]:
!python code/hybrid_ids_cicids2017.py
!python code/knowledge_distillation.py

## Bước 6: Cập nhật hình ảnh và bảng LaTeX

Sau khi huấn luyện xong, các script dưới đây tạo biểu đồ và cập nhật số liệu vào `sections/results.tex`.
Kết quả được lưu trên Drive nên bạn có thể tải về hoặc mở trực tiếp từ Drive.

In [ ]:
!python code/replot_figures.py
!python code/populate_latex_tables.py